# Расхождения commission_monthly по agr_id (Q1 2026)

Отдельная тетрадка для поиска `agr_id`, где `commission_monthly` из Озера (R2 `tariff_fix.c_summa`) отличается от Excel.

Итоговые поля:
- `report_month`
- `agr_id`
- `commission_monthly_lake`
- `commission_monthly_excel`
- `tariff_name`

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

In [ ]:
excel_sources = [
    {'report_month': '2026-01-01', 'path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx', 'header': 1},
    {'report_month': '2026-02-01', 'path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx', 'header': 0},
    {'report_month': '2026-03-01', 'path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx', 'header': 0},
]

output_csv_path = './q1_commission_monthly_mismatch_by_agr.csv'

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 1) Загрузка Excel Q1 и нормализация commission_monthly

In [ ]:
def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    norm_map = {normalize_colname(c): c for c in columns}
    for alias in aliases:
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    return None


def normalize_agr_id(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return None
    return s


frames = []
for src in excel_sources:
    raw = pd.read_excel(src['path'], header=src['header'])

    agr_col = pick_column(raw.columns, ['ID договора', 'agr_id'])
    cm_col = pick_column(raw.columns, ['Комиссия \n(₽ в месяц)', 'Комиссия (₽ в месяц)'])

    if agr_col is None:
        raise ValueError(f"В файле {src['path']} не найдена колонка agr_id (ID договора)")
    if cm_col is None:
        raise ValueError(f"В файле {src['path']} не найдена колонка комиссии: Комиссия \\n(₽ в месяц)")

    df = pd.DataFrame({
        'agr_id': raw[agr_col].apply(normalize_agr_id),
        'commission_monthly_excel': pd.to_numeric(raw[cm_col], errors='coerce'),
    })
    df['report_month'] = pd.to_datetime(src['report_month'])
    frames.append(df)

excel_df = pd.concat(frames, ignore_index=True)
excel_df = excel_df[excel_df['agr_id'].notna()].copy()

print(f'Excel Q1 rows (with agr_id): {len(excel_df):,}')
print('Rows by month:')
print(excel_df.groupby(excel_df['report_month'].dt.strftime('%Y-%m')).size())
excel_df.head(5)

## 2) Выгрузка commission_monthly и tariff_name из Озера (R2)

In [ ]:
agr_values = sorted(excel_df['agr_id'].astype(str).unique().tolist())
agr_in = ', '.join([f"'{x}'" for x in agr_values]) if agr_values else "''"

sql_lake = f"""
select
  cast(m.id as string) as agr_id,
  cast(max(tp.c_name) as string) as tariff_name,
  max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_lake
from ods.scd1_z_r2_ip_merchants m
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_tune tt
  on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_fix tf
  on cast(tf.id as string) = cast(tt.c_tariff as string)
where cast(m.id as string) in ({agr_in})
group by cast(m.id as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    lake_df = imp.fetch(sql_lake)

if lake_df is None:
    lake_df = pd.DataFrame(columns=['agr_id', 'tariff_name', 'commission_monthly_lake'])

if not lake_df.empty:
    lake_df['agr_id'] = lake_df['agr_id'].astype(str)
    lake_df['tariff_name'] = lake_df['tariff_name'].astype('string')
    lake_df['commission_monthly_lake'] = pd.to_numeric(lake_df['commission_monthly_lake'], errors='coerce')

print(f'Lake rows: {len(lake_df):,}')
lake_df.head(5)

## 3) Строгое сравнение и отбор расхождений

In [ ]:
rows_excel = len(excel_df)

merged_df = excel_df.merge(lake_df, on='agr_id', how='left')
rows_after_merge = len(merged_df)

excel_val = merged_df['commission_monthly_excel']
lake_val = merged_df['commission_monthly_lake']

is_diff_strict = (
    (excel_val.isna() & lake_val.notna())
    | (excel_val.notna() & lake_val.isna())
    | (excel_val.notna() & lake_val.notna() & (excel_val != lake_val))
)

merged_df['is_diff_strict'] = is_diff_strict

mismatch_df = merged_df.loc[merged_df['is_diff_strict'], [
    'report_month',
    'agr_id',
    'commission_monthly_lake',
    'commission_monthly_excel',
    'tariff_name',
]].copy()

mismatch_df['report_month'] = pd.to_datetime(mismatch_df['report_month'], errors='coerce').dt.strftime('%Y-%m')
mismatch_df = mismatch_df.sort_values(['report_month', 'agr_id']).reset_index(drop=True)

print('QC:')
print(f'  rows in excel perimeter: {rows_excel:,}')
print(f'  rows after merge:        {rows_after_merge:,}')
print(f'  strict mismatches:       {len(mismatch_df):,}')

missing_lake_cnt = merged_df.loc[merged_df['commission_monthly_lake'].isna(), 'agr_id'].nunique()
print(f'  unique agr_id without lake commission: {missing_lake_cnt:,}')

print('\nMismatches by month:')
print(mismatch_df.groupby('report_month').size().rename('mismatch_rows'))

# Проверка на максимальное значение c_summa (commission_monthly_lake) из Озера
if merged_df['commission_monthly_lake'].notna().any():
    max_idx = merged_df['commission_monthly_lake'].idxmax()
    max_row = merged_df.loc[max_idx, ['report_month', 'agr_id', 'commission_monthly_lake', 'commission_monthly_excel', 'tariff_name']]
    max_row = max_row.copy()
    max_row['report_month'] = pd.to_datetime(max_row['report_month'], errors='coerce').strftime('%Y-%m')

    print('\nMax c_summa from lake:')
    print(max_row.to_frame().T)
else:
    print('\nMax c_summa from lake: not found (all values are NULL)')

# 5 примеров agr_id, где значение из Озера совпадает с Excel (оба значения не NULL)
matches_df = merged_df.loc[
    merged_df['commission_monthly_lake'].notna()
    & merged_df['commission_monthly_excel'].notna()
    & (merged_df['commission_monthly_lake'] == merged_df['commission_monthly_excel']),
    ['report_month', 'agr_id', 'commission_monthly_lake', 'commission_monthly_excel', 'tariff_name']
].copy()

matches_df['report_month'] = pd.to_datetime(matches_df['report_month'], errors='coerce').dt.strftime('%Y-%m')
matches_df = matches_df.sort_values(['report_month', 'agr_id']).reset_index(drop=True)

print('\nПримеры 5 agr_id с совпадением Озеро=Excel:')
print(matches_df.head(5))

mismatch_df.head(20)

## 4) Выгрузка результата

In [ ]:
output_path = Path(output_csv_path)
matched_sample_path = Path('./q1_commission_monthly_matches_sample5.csv')

mismatch_df.to_csv(output_path, index=False, encoding='utf-8-sig')
matches_df.head(5).to_csv(matched_sample_path, index=False, encoding='utf-8-sig')

print(f'CSV saved (mismatch): {output_path.resolve()}')
print(f'CSV saved (5 matches sample): {matched_sample_path.resolve()}')

mismatch_df.head(5)